# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("C:/Users/jihun/OneDrive/Desktop/DS 실습자료/LV2/실전 머신러닝/데이터/creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727



Train 크기: (227845, 28)
Test 크기: (56962, 28)


---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

In [2]:
# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.
majority_pred = np.zeros(len(y_test), dtype=int)
majority_accuracy = accuracy_score(y_test, majority_pred)
majority_recall = recall_score(y_test, majority_pred)

print("Accuracy:", majority_accuracy)
print("Recall:", majority_recall)

Accuracy: 0.9982795547909132
Recall: 0.0


### 결과
### 
|실제 정답| 정상 0으로 예측 | 사기 1로 예측 | 내용 |
|---|---|---|---|
|정상 0 | TN = 56,864 | FP = 0| 정상 거래는 전부 맞춤 |
|사기 1 |FN = 98| TP = 0| 사기 거래는 전부다 틀림|

### -> TN은 정상을 정상으로 맞춘 건수, TP는 사기를 사기로 맞친 건수
### -> FP는 정상인데 사기라고 잘못 예측한 오탐
### -> FN은 사기인데 정상이라고 잘못 예측한 미탐

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?
### 데이터의 대부분이 정상거래이기 때문에 전부 다 정상거래로 예측을 해도 정확도(Accuracy)는 높음
### 하지만 사기를 탐지해야하는 데 Accurcay 하나만으로는 이 모델의 성능을 평가할 수 없음
### Recall, 정밀도 등도 같이 확인해야 모델의 성능을 평가할 수 있음

### 불균형 데이터에서는 Accurcay가 높아도 소수 클래스를 전혀 탐지 못할 수 있다.

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [3]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.
base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train, y_train)
base_pred = base_model.predict(X_test)
base_prob = base_model.predict_proba(X_test)[:, 1]

base_accuracy = accuracy_score(y_test, base_pred)
base_precision = precision_score(y_test, base_pred)
base_recall = recall_score(y_test, base_pred)
base_f1 = f1_score(y_test, base_pred)
base_pr_auc = average_precision_score(y_test, base_prob)

print("Accuracy:", base_accuracy)
print("Precision:", base_precision)
print("Recall:", base_recall)
print("F1-score:", base_f1)
print("PR-AUC:", base_pr_auc)

print("\n[Confusion Matrix]")
print(confusion_matrix(y_test, base_pred))

Accuracy: 0.9991573329588147
Precision: 0.8289473684210527
Recall: 0.6428571428571429
F1-score: 0.7241379310344828
PR-AUC: 0.7435600673915171

[Confusion Matrix]
[[56851    13]
 [   35    63]]


### 혼동 행렬
|실제 정답 / 예측| 정상 0 | 사기 | 실제 거래 합계|
|---|---|---|---|
|정상 0|TN = 56,851|FP = 13| 56,864|
|사기 1|FN = 35| TP = 63| 98|
|예측 합계|56,886|76|56,962|

### TP : 사기 사건 63건을 사기라고 올바르게 예측한 경우
### FN : 사기 35건을 정상이라고 잘못 예측한 경우
### FP : 정상 13건을 사기라고 잘못 예측한 경우
### TN : 정상 56,851건을 정상이라고 올바르게 예측한 경우

### 사기라고 예측한 거래는 TP + FP = 76건
### 실제 사기 거래는 TP + FN = 98건

|지표|계산 식|결과|내용|
|---|---|---|---|
|Accurcay|(56,851 + 63) / 56,962| 0.999|전체 거래의 약 99.9%를 올바르게 분류|
|Precision|63 / (63 + 13)|0.828|사기라고 예측한 76건 중 약 82.8%가 실제 사기|
|Recall|63 / (63+35)| 0.642|실제 사기 98건 중 64.2%정도를 올바르게 분류|
|F1|2 * 63 / (2 * 63 + 13 + 35)|0.724|Precison와 Recall의 조화 평균|
|PR-AUC|정답과 사기 점수|0.743|Precions-Recall 성능 요약|

### Precision는 사기라고 말한 것을 얼마나 믿을 수 있는 지?
### Recall은 사기를 실제로 얼마나 놓치지 않고 찾았는 지? 
### 해석 : 모두 정상으로 예측한 기준선은 0건을 탐지해냄, 기본 모델같은 경우 63건을 찾고 35건을 놓쳤으며
### 정상 13건을 사기로 잘못 예측함
### 사기를 사기로 탐지하는 능력은 있고 정확도도 매우 높은 편이지만
### Recall이 64%정도 이기 때문에 사기를 거의 다 잡아낼 수 있는 모델이다 라고 평가할 수는 없다.

### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?
### Recall -> 실제 사기중에서 얼마나 사기를 정확하게 찾아 냈는 지를 나타내기 때문
### 사기를 놓치는 FN을 중요하게 생각하신다면 Recall값을 최대한 높여서 만들어야함



---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [4]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.
weighted_model = LogisticRegression(class_weight="balanced", max_iter=1000)
weighted_model.fit(X_train, y_train)
weighted_pred = weighted_model.predict(X_test)
weighted_prob = weighted_model.predict_proba(X_test)[:, 1]
weighted_precision = precision_score(y_test, weighted_pred)

weighted_recall = recall_score(y_test, weighted_pred)
weighted_f1 = f1_score(y_test, weighted_pred)
weighted_pr_auc = average_precision_score(y_test, weighted_prob)

print("Precision:", weighted_precision)
print("Recall:", weighted_recall)
print("F1-score:", weighted_f1)
print("PR-AUC:", weighted_pr_auc)

Precision: 0.05787781350482315
Recall: 0.9183673469387755
F1-score: 0.1088929219600726
PR-AUC: 0.7092143497034353


### balanced는 어떤 가중치를 부여하는 지?
### 클래스 K의 학습 표본수가 NK 전체 학습 표본수가 N 클래스 수가 K면
$$w_k \frac{n}{K n_k}$$
### 가중치가 0.50정도 사기 한건이 약 289.14 정도의 수치
### 같은 정도의 손실이라면 사기 한건이 정상 한건보다 약 577.3배 큰 비중을 가진다. 

### Precision 처음에 82% -> 지금 5% -> 급격하게 사기라고 예측한 것중에 실제 사기였던 비율이 크게 낮아짐
### Recall 처음에 64% -> 지금 91% -> 실제 사기를 기존보다 더 잘 예측
### F1 처음에 72% -> 지금 10% -> Precision의 값이 매우 작아 종합값이 매우 떨어짐

### TP : 기존 모델 63건, 클래스 가중치 부여한 경우 90건 -> 사기 사건 27건 추가탐지
### FN : 기존 모델 35건, 클래스 가중치 부여한 경우 8건 -> 사기라고 탐지 못한 경우 27건 감소
### FP : 기존 모델 13건, 클래스 가중치 부여한 경우 1,465건 -> 오탐 1,452건 증가
### TP + FP : 기존모델 76건, 클래스 가중치 부여한 경우 1,555 -> 대부분 사기라고 예측을 해버림

### 해석 : 모두 정상으로 예측한 기준선은 사기를 0건 찾았지만, 기본 모델은 63건을 찾고 35건을 놓쳤고
### 정상 13건을 오탐했다. 사기 탐지 능력은 있으나 Accurcay가 99%를 넘는다고 매우 좋은 모델이다라고 해석할 수는 없다.
### 실제 사기의 약 35%를 놓치기 때문이다.


## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [5]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.
under_sampler = RandomUnderSampler(random_state=42)
X_under, y_under = under_sampler.fit_resample(X_train, y_train)

print("[리샘플링 전]")
print(y_train.value_counts())

print("\n[리샘플링 후]")
print(y_under.value_counts())

under_model = LogisticRegression(max_iter=1000)
under_model.fit(X_under, y_under)
under_pred = under_model.predict(X_test)
under_prob = under_model.predict_proba(X_test)[:, 1]

print("\nPrecision:", precision_score(y_test, under_pred))
print("Recall:", recall_score(y_test, under_pred))
print("F1-score:", f1_score(y_test, under_pred))
print("PR-AUC:", average_precision_score(y_test, under_prob))

[리샘플링 전]
Class
0    227451
1       394
Name: count, dtype: int64

[리샘플링 후]
Class
0    394
1    394
Name: count, dtype: int64

Precision: 0.03759398496240601
Recall: 0.9183673469387755
F1-score: 0.07223113964686999
PR-AUC: 0.635818598713662


### 왜 언더 샘플링방법을 사용하는 가?
### 댜수 클래스의 수를 줄이면 학습 자료에서 사기 표본의 상대적 비중이 높아짐
### 처리할 자료의 양이 줄어서 학습 속도 향상을 기대할 수 있음
### 다만, 현재 무작위로 남긴 394건이 정상 거래의 다양한 패턴을 충분히 반영하지 못하는 문제가 발생할 수 있음

### 샘플링 비율, 남기는 표본, 모델 설정을 다르게하면 모델의 결과가 달라질 수 있다.
### 이번 언더샘플링은 사기 탐지(Recall)의 값은 높았지만 오탐이 매우 많이 발생했다.
### 언더샘플링 자체가 잘못된 것이 아니라 데이터셋이 언더샘플링에 부적절했다고 판단한다.


## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [6]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("[리샘플링 전]")
print(y_train.value_counts())

print("\n[SMOTE 후]")
print(y_smote.value_counts())

smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_smote, y_smote)
smote_pred = smote_model.predict(X_test)
smote_prob = smote_model.predict_proba(X_test)[:, 1]

print("\nPrecision:", precision_score(y_test, smote_pred))
print("Recall:", recall_score(y_test, smote_pred))
print("F1-score:", f1_score(y_test, smote_pred))
print("PR-AUC:", average_precision_score(y_test, smote_prob))

[리샘플링 전]
Class
0    227451
1       394
Name: count, dtype: int64

[SMOTE 후]
Class
0    227451
1    227451
Name: count, dtype: int64

Precision: 0.05555555555555555
Recall: 0.9183673469387755
F1-score: 0.10477299185098952
PR-AUC: 0.718834211008162


### SMOTE : 소수 클래스의 기존 표본과 가까운 클래스 이웃을 합성하여 표본을 만든다.
### -> 사기행을 중복하여 생성하는 방식이 아님
### -> 사기 394건을 참고하여 합성한 학습자료

### 가중치, 언더샘플링, SMOTE 비교
|처리|TP|FN|FP|전체 사기 예측|
|---|---|---|---|---|
|Class Weight|90|8|1,465|1,555|
|Random Undersampling|90|8|2,304|2,394|
|SMOTE|90|8|1,530|1,620|

### -> Class Weight가 SMOTE보다 오탐이 65건이 더 적었고, AP는 SMOTE가 약 0.718로 Class Weight보다 약 0.09높다

### SMOTE의 한계점
### 가까운 사기 표본 사이의 실제 사기 분포를 잘 나타낸다면 도움이 될 가능성이 높음
### 그러나 이상치 주변이나 경계가 모호한 경우 의도치 않은 합성 표본이 발생할 수 있음
### 합성 표본을 많이 만드는 것이 모든 성능평가 지표의 상승을 의미하지는 않음

### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?
### Test 데이터는 실제 배포 환경과 유사한 상황을 가정하고 실시간으로 들어오는 데이터들이 많다.
### 그런데 인위적으로 리샘플링을 진행한다면 기존에 실제 환경과 다른 데이터셋들이 발생할 수 있다.
### 과적합이 발생할 가능성이 있다.

### UnderSampling과 SMOTH는 반드시 Train 데이터에만 적용한다.

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [ ]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.

### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  
여기에 작성하세요.

### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**  
여기에 작성하세요.

### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:**
여기에 작성하세요.


### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.

**답변:**
여기에 작성하세요.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?
-> 다수 클래스만 잘 맞춰도 Accuracy가 높아질 수 있기 때문이다.

2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?
-> 양성이라고 예측한 것 중에 실제 양성 비율

3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?
-> 실제 양성 중에서 모델이 정확히 양성으로 판별한 비율

4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?
-> class_weight -> 소수 클래스 오류에 더 큰 가중치
-> Undersampling -> 다수 클래스 데이터 감소
-> SMOTH -> 소수 클래스 합성 데이터 증가

5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?
-> Train 데이터셋에만 적용

6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?
-> Recall은 높아지고 Precision은 낮아지는 경향이 있다.